# Nhận diện biển báo giao thông bằng mạng CNN

Notebook gồm toàn bộ quá trình: huấn luyện mô hình CNN (theo `traffic_sign.py`) trên bộ dữ liệu GTSRB, kiểm tra trên tập Test, chuyển sang TensorFlow Lite cho ứng dụng và thử nghiệm với 100 ảnh biển báo chụp tại Việt Nam.

Cách chạy: Runtime → Change runtime type → T4 GPU → Save, sau đó Runtime → Run all (khoảng 20 phút). Khi chạy xong, file `ket_qua.zip` được tải về máy.

In [ ]:
# Bước 1: Cài thư viện và kiểm tra GPU
!pip install -q kagglehub
import os, glob, time, random, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import kagglehub
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Bước 2: Tải bộ dữ liệu GTSRB
duong_dan = kagglehub.dataset_download('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign')
ROOT = None
for d, thu_muc_con, cac_file in os.walk(duong_dan):
    if 'Train' in thu_muc_con and 'Test.csv' in cac_file:
        ROOT = d
        break
print('Thư mục GTSRB:', ROOT)

In [ ]:
# Bước 3: Đọc ảnh và nhãn (giống traffic_sign.py)
data = []
labels = []
classes = 43
for i in range(classes):
    path = os.path.join(ROOT, 'Train', str(i))
    for a in os.listdir(path):
        try:
            image = Image.open(os.path.join(path, a)).convert('RGB')
            image = image.resize((30, 30))
            data.append(np.array(image))
            labels.append(i)
        except Exception:
            print('Lỗi đọc ảnh:', a)
data = np.array(data)
labels = np.array(labels)
print(data.shape, labels.shape)

In [ ]:
# Bước 4: Chia dữ liệu, xây dựng mô hình CNN và huấn luyện (giống traffic_sign.py)
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout

X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
y_train = to_categorical(y_train, 43)
y_test = to_categorical(y_test, 43)

model = Sequential()
model.add(Conv2D(filters=32, kernel_size=(5, 5), padding='same', activation='relu', input_shape=X_train.shape[1:]))
model.add(Conv2D(filters=32, kernel_size=(5, 5), padding='same', activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(rate=0.5))
model.add(Dense(43, activation='softmax'))
model.summary()

epochs = 20
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
bat_dau = time.time()
history = model.fit(X_train, y_train, batch_size=32, epochs=epochs, validation_data=(X_test, y_test))
thoi_gian_huan_luyen = (time.time() - bat_dau) / 60
model.save('traffic_classifier.h5')

In [ ]:
# Bước 5: Vẽ đồ thị độ chính xác và hàm mất mát
ep = range(1, epochs + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, history.history['accuracy'], 'o-', label='Huấn luyện')
ax[0].plot(ep, history.history['val_accuracy'], 's-', label='Kiểm định')
ax[0].set_title('Độ chính xác')
ax[0].set_xlabel('Epoch')
ax[0].legend()
ax[0].grid(alpha=0.3)
ax[1].plot(ep, history.history['loss'], 'o-', label='Huấn luyện')
ax[1].plot(ep, history.history['val_loss'], 's-', label='Kiểm định')
ax[1].set_title('Hàm mất mát')
ax[1].set_xlabel('Epoch')
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('do_thi_huan_luyen.png', dpi=200)
plt.show()

In [ ]:
# Bước 6: Kiểm tra trên 8.000 ảnh của tập Test và tính Precision, Recall, F1
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
test = pd.read_csv(os.path.join(ROOT, 'Test.csv'))
test = test.sample(min(8000, len(test)), random_state=42)
X8 = np.array([np.array(Image.open(os.path.join(ROOT, p)).convert('RGB').resize((30, 30))) for p in test['Path']])
y8 = test['ClassId'].values
du_doan8 = model.predict(X8, verbose=0).argmax(axis=1)
do_chinh_xac = accuracy_score(y8, du_doan8)
P, R, F1, so_anh = precision_recall_fscore_support(y8, du_doan8, labels=list(range(43)), zero_division=0)
bang_f1 = pd.DataFrame({'Nhãn': range(43), 'Số ảnh': so_anh, 'Precision': P.round(2), 'Recall': R.round(2), 'F1': F1.round(2)})
bang_f1.to_csv('bang_precision_recall_f1.csv', index=False)
print(bang_f1.to_string(index=False))
print('Độ chính xác:', round(do_chinh_xac * 100, 2), '%')

In [ ]:
# Bước 7: Chuyển mô hình sang TensorFlow Lite để chạy trong app
# Dựng lại mô hình với batch = 1 rồi chép trọng số sang, chuyển ở dạng float32
# (bản lượng tử hóa không chạy được trong app vì thư viện TFLite của app chưa hỗ trợ phiên bản toán tử mới)
m1 = Sequential()
m1.add(tf.keras.layers.Input((30, 30, 3), batch_size=1))
for layer in model.layers:
    m1.add(layer.__class__.from_config(layer.get_config()))
m1.set_weights(model.get_weights())
open('model.tflite', 'wb').write(tf.lite.TFLiteConverter.from_keras_model(m1).convert())

it = tf.lite.Interpreter(model_path='model.tflite')
it.allocate_tensors()
VAO = it.get_input_details()[0]['index']
RA = it.get_output_details()[0]['index']
it.set_tensor(VAO, X8[:1].astype(np.float32))
it.invoke()
chenh_lech = np.abs(it.get_tensor(RA) - model.predict(X8[:1], verbose=0)).max()
print('Kích thước model.tflite:', round(os.path.getsize('model.tflite') / 2**20, 2), 'MB')
print('Chênh lệch so với mô hình Keras:', chenh_lech)

In [ ]:
# Bước 8: Thử nghiệm 100 ảnh biển báo chụp ở Việt Nam
# Dùng model.tflite và xử lý giống app: cắt khung vuông quanh biển, chạy 5 vùng cắt lệch nhau,
# đưa về 30x30, lấy trung bình xác suất, ngưỡng tin cậy 60%.
# Nhóm A: biển giống lớp GTSRB. Nhóm B: biển cảnh báo nền vàng. Nhóm C: biển không có trong 43 lớp.
vn = kagglehub.dataset_download('maitam/vietnamese-traffic-signs')
thu_muc_vn = os.path.dirname(glob.glob(vn + '/**/classes.txt', recursive=True)[0])
ten_lop = [l.strip() for l in open(os.path.join(thu_muc_vn, 'classes.txt'), encoding='utf-8') if l.strip()]

# Bảng ghép mã biển Việt Nam với nhãn GTSRB tương ứng
GHEP = {'P.102': 17, 'P.127*50': 2, 'P.127*60': 3, 'P.127*80': 5, 'P.106a*Xe tải': 16, 'R.302a': 38, 'R.303': 40,
        'W.201a': 19, 'W.201b': 20, 'W.203c': 24, 'W.207': 11, 'W.221b': 22, 'W.224': 27, 'W.225': 28, 'W.233': 18}
CANH_BAO = [11] + list(range(18, 32))

mau = []
for f in glob.glob(os.path.join(thu_muc_vn, 'labels', '*.txt')):
    anh = os.path.join(thu_muc_vn, 'images', os.path.basename(f)[:-4] + '.jpg')
    if not os.path.exists(anh):
        continue
    for dong in open(f):
        v = dong.split()
        if len(v) < 5:
            continue
        lop = ten_lop[int(v[0])]
        if lop in GHEP:
            nhom = 'B' if GHEP[lop] in CANH_BAO else 'A'
        elif lop[0] in 'PRWI':
            nhom = 'C'
        else:
            continue   # bỏ biển phụ, camera...
        mau.append([anh, lop, nhom, GHEP.get(lop, -1)] + [float(x) for x in v[1:5]])
mau = pd.DataFrame(mau, columns=['anh', 'lop', 'nhom', 'nhan', 'cx', 'cy', 'w', 'h'])
chon = pd.concat([mau[mau['nhom'] == 'A'].sample(70, random_state=42),
                  mau[mau['nhom'] == 'B'].sample(15, random_state=42),
                  mau[mau['nhom'] == 'C'].sample(15, random_state=42)]).sample(frac=1, random_state=42).reset_index(drop=True)

CAT = [(1.0, 0, 0), (0.9, 0, 0), (1.1, 0, 0), (1.0, 0.05, 0.05), (1.0, -0.05, -0.05)]

def cat_vuong(im, x0, y0, d):
    # cắt khung vuông cạnh d, không vượt ra ngoài ảnh (giống app)
    x0 = min(max(0, x0), im.width - 1)
    y0 = min(max(0, y0), im.height - 1)
    return im.crop((round(x0), round(y0), round(min(im.width, x0 + d)), round(min(im.height, y0 + d))))
ket_qua = []
for k, r in enumerate(chon.itertuples(), 1):
    im = Image.open(r.anh).convert('RGB')
    W, H = im.size
    cx, cy = r.cx * W, r.cy * H
    canh = max(r.w * W, r.h * H) * 1.2
    xs = []
    tg = 0
    for ti_le, dx, dy in CAT:
        d = canh * ti_le
        x0, y0 = cx + dx * canh - d / 2, cy + dy * canh - d / 2
        o = cat_vuong(im, x0, y0, d).resize((30, 30), Image.BICUBIC)
        t = time.perf_counter()
        it.set_tensor(VAO, np.array(o, np.float32)[None])
        it.invoke()
        xs.append(it.get_tensor(RA)[0])
        tg += (time.perf_counter() - t) * 1000
    p = np.mean(xs, axis=0)
    ket_qua.append({'Lần': k, 'Ảnh': os.path.basename(r.anh), 'Biển': r.lop, 'Nhóm': r.nhom, 'Nhãn đúng': r.nhan,
                    'Dự đoán': int(p.argmax()), 'Độ tin cậy (%)': round(float(p.max()) * 100, 1),
                    'Vượt 60%': p.max() >= 0.6, 'Đúng': (int(p.argmax()) == r.nhan) if r.nhom != 'C' else None,
                    'Thời gian (ms)': round(tg, 2)})
kq = pd.DataFrame(ket_qua)
kq.to_csv('ket_qua_100_anh.csv', index=False, encoding='utf-8-sig')

print('Kết quả theo nhóm:')
for g in ['A', 'B', 'C']:
    d = kq[kq['Nhóm'] == g]
    if g == 'C':
        print(f'  Nhóm C: {len(d)} ảnh, báo "chưa chắc chắn" {int((~d["Vượt 60%"]).sum())} lần')
    else:
        print(f'  Nhóm {g}: {len(d)} ảnh, đúng {int(d["Đúng"].sum())} ({d["Đúng"].mean() * 100:.1f}%), vượt 60%: {int(d["Vượt 60%"].sum())}')
print('Kết quả nhóm A theo loại biển:')
a = kq[kq['Nhóm'] == 'A']
print(a.groupby('Biển').agg(so_lan=('Đúng', 'size'), dung=('Đúng', 'sum'), tin_cay_tb=('Độ tin cậy (%)', 'mean')).round(1).to_string())

fig, axs = plt.subplots(10, 10, figsize=(14, 15))
for ax, r, (_, k) in zip(axs.flat, chon.itertuples(), kq.iterrows()):
    im = Image.open(r.anh).convert('RGB')
    W, H = im.size
    canh = max(r.w * W, r.h * H) * 1.2
    x0, y0 = r.cx * W - canh / 2, r.cy * H - canh / 2
    ax.imshow(cat_vuong(im, x0, y0, canh).resize((96, 96)))
    mau_vien = 'gray' if k['Nhóm'] == 'C' else ('green' if k['Đúng'] else 'red')
    ax.set_title(f"{k['Lần']} {k['Nhóm']} {k['Độ tin cậy (%)']:.0f}%", fontsize=8, color=mau_vien)
    ax.set_xticks([])
    ax.set_yticks([])
    for vien in ax.spines.values():
        vien.set_color(mau_vien)
        vien.set_linewidth(3)
plt.tight_layout()
plt.savefig('hinh_100_anh.png', dpi=150)
plt.show()

In [ ]:
# Bước 9: Số liệu cho báo cáo và tải file về
ab = kq[kq['Nhóm'] != 'C']
vuot = ab[ab['Vượt 60%']]
print('SỐ LIỆU CHO BÁO CÁO')
print('Ngày chạy:', time.strftime('%d/%m/%Y'))
print('Huấn luyện:', len(X_train), 'ảnh | kiểm định:', len(X_test), 'ảnh |', epochs, 'epoch | thời gian:', round(thoi_gian_huan_luyen, 1), 'phút')
print('Epoch cuối: độ chính xác huấn luyện', round(history.history['accuracy'][-1] * 100, 2), '% | kiểm định', round(history.history['val_accuracy'][-1] * 100, 2), '%')
print('Tập Test 8.000 ảnh: đúng', int((y8 == du_doan8).sum()), 'ảnh | độ chính xác', round(do_chinh_xac * 100, 2), '%')
print('Macro Precision / Recall / F1:', round(P.mean(), 2), '/', round(R.mean(), 2), '/', round(F1.mean(), 2))
print('Số lớp có F1 >= 0,95:', int((F1 >= 0.95).sum()), '/ 43')
print('model.tflite:', round(os.path.getsize('model.tflite') / 2**20, 2), 'MB')
print('100 ảnh Việt Nam: nhóm A+B đúng', int(ab['Đúng'].sum()), '/', len(ab), '| khi vượt 60% đúng', int(vuot['Đúng'].sum()), '/', len(vuot))

with zipfile.ZipFile('ket_qua.zip', 'w') as z:
    for f in ['model.tflite', 'traffic_classifier.h5', 'do_thi_huan_luyen.png', 'bang_precision_recall_f1.csv', 'ket_qua_100_anh.csv', 'hinh_100_anh.png']:
        z.write(f)
from google.colab import files
files.download('ket_qua.zip')